# Customer Churn Prediction — Data Preprocessing and Train/Validation/Test Split

This notebook implements **Step 4: Data Preprocessing and Train/Validation/Test Split** for the Customer Churn Prediction project.

### Scope & Workflow:
```text
Raw Dataset (data/raw/telco_customer_churn.csv)
    ↓
Deterministic Cleaning (TotalCharges whitespace coercion to 0.0, duplicate checks)
    ↓
Target Preparation (Churn: 'No' -> 0, 'Yes' -> 1)
    ↓
Feature / Target Separation (Drop target & customerID)
    ↓
Train / Validation / Test Split (70% Train, 15% Val, 15% Test — Stratified by Churn)
    ↓
Preprocessing Pipeline (ColumnTransformer)
    ├── Numerical: SimpleImputer(median) + StandardScaler
    └── Categorical: SimpleImputer(most_frequent) + OneHotEncoder(handle_unknown='ignore')
    ↓
Fit ONLY on X_train (Zero Data Leakage)
    ↓
Transform X_train, X_val, X_test
    ↓
Save Preprocessing Artifact (models/preprocessor.joblib) & Processed Partitions
```

## 1. Objective

The primary objective of this phase is to transform raw customer observations into clean, scaled, model-ready numeric arrays while strictly guaranteeing **zero data leakage**.

**Core Requirements:**
1. **Preserve Raw Data**: `data/raw/telco_customer_churn.csv` remains strictly immutable.
2. **Evidence-Based Cleaning**: Coerce 11 whitespace records in `TotalCharges` (identified during EDA for accounts with `tenure == 0`) to `0.0` and convert to `float64`.
3. **Target Validation & Separation**: Map binary `Churn` (`No` → 0, `Yes` → 1) and exclude non-predictive `customerID`.
4. **Stratified Split**: Partition observations into 70% Training, 15% Validation, and 15% Testing while preserving identical churn proportions (~26.54%).
5. **Modular ColumnTransformer**: Impute and standardize numerical columns; impute and one-hot encode categorical columns with unseen category resilience (`handle_unknown='ignore'`).
6. **Strict Leakage Prevention**: Fit pipeline exclusively on $X_{train}$.
7. **Artifact Persistence**: Save fitted preprocessor (`models/preprocessor.joblib`) for reuse in subsequent model training and production inference.

## 2. Load Dataset

Importing required dependencies and loading the raw Telco customer churn dataset using `src.churn_prediction.data.loader.load_dataset`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.churn_prediction.config import (
    PREPROCESSOR_PATH,
    PROCESSED_DATA_DIR,
    RANDOM_STATE,
    RAW_DATA_PATH,
)
from src.churn_prediction.data.loader import load_dataset
from src.churn_prediction.features.preprocessing import (
    build_preprocessing_pipeline,
    clean_dataset,
    fit_preprocessor,
    get_feature_names,
    identify_feature_types,
    load_preprocessor,
    prepare_target,
    save_preprocessor,
    save_processed_datasets,
    separate_features_target,
    split_data,
    transform_data,
)

print("Project root:", PROJECT_ROOT)
df_raw = load_dataset(RAW_DATA_PATH)
df_raw[
    [
        "customerID",
        "gender",
        "SeniorCitizen",
        "Partner",
        "Dependents",
        "tenure",
        "PhoneService",
        "MonthlyCharges",
        "TotalCharges",
        "Churn",
    ]
].head()

Project root: /home/lenovo/Documents/customer churn prediction
Successfully loaded raw dataset: 7,043 rows, 21 columns.


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,70.70,151.65,Yes


## 3. Review EDA Findings

Key findings from Steps 2 and 3 that dictate our preprocessing design:

1. **Target Distribution**: `Churn` has 5,174 `No` (73.46%) and 1,869 `Yes` (26.54%). Imbalance ratio is ~2.77:1. **Stratified splitting** is mandatory to prevent partition class drift.
2. **Whitespace Anomaly in TotalCharges**: 11 records have `TotalCharges == ' '`. All 11 records belong to new customers with `tenure == 0`. Since no billing cycle has completed, their true accumulated total charge is `$0.00`.
3. **Customer Identifier**: `customerID` has 7,043 unique alphanumeric strings (100% unique). It has zero generalizable predictive signal and must be excluded from feature matrices.
4. **No Duplicate Rows**: Exactly 0 duplicate customer records exist.
5. **Feature Modalities**: 4 numerical features (`tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`) and 15 categorical features (service subscriptions, contract terms, billing options).

## 4. Data Cleaning

We apply `clean_dataset(df)` from `src.churn_prediction.features.preprocessing`.
This function:
- Creates a defensive copy (leaving `data/raw/` untouched).
- Strips whitespace and coerces `TotalCharges` to `0.0` for zero-tenure accounts.
- Converts `TotalCharges` from string/object to `float64`.
- Verifies and handles duplicates.

In [2]:
df_clean = clean_dataset(df_raw)

print(f"Cleaned dataset dimensions: {df_clean.shape}")
print(f"TotalCharges dtype before cleaning: {df_raw['TotalCharges'].dtype}")
print(f"TotalCharges dtype after cleaning:  {df_clean['TotalCharges'].dtype}")
print(f"TotalCharges unresolved nulls:      {df_clean['TotalCharges'].isnull().sum()}")

zero_tenure_sample = df_clean.loc[
    df_clean["tenure"] == 0, ["tenure", "MonthlyCharges", "TotalCharges"]
].head()
print("Accounts with tenure == 0 TotalCharges sample:")
print(zero_tenure_sample)

Cleaned dataset dimensions: (7043, 21)
TotalCharges dtype before cleaning: object
TotalCharges dtype after cleaning:  float64
TotalCharges unresolved nulls:      0
Accounts with tenure == 0 TotalCharges sample:
    tenure  MonthlyCharges  TotalCharges
488      0           52.55           0.0
753      0           20.25           0.0
936      0           80.85           0.0
1082     0           25.75           0.0
1340     0           56.05           0.0


## 5. Target Preparation

Convert the target variable `Churn` into a clean binary representation:
- `No` → `0` (Negative class / retained customer)
- `Yes` → `1` (Positive class / churned customer)

We validate that the target contains strictly binary classes without missing values.

In [3]:
y = prepare_target(df_clean, target_col="Churn")

print("Target variable validation:")
print(f"Class 0 (Retained / No): {(y == 0).sum():,} ({(y == 0).mean() * 100:.2f}%)")
print(f"Class 1 (Churned / Yes): {(y == 1).sum():,} ({(y == 1).mean() * 100:.2f}%)")
print(f"Overall churn rate: {y.mean() * 100:.2f}%")
print(f"Missing values in target: {y.isnull().sum()}")

Target variable validation:
Class 0 (Retained / No): 5,174 (73.46%)
Class 1 (Churned / Yes): 1,869 (26.54%)
Overall churn rate: 26.54%
Missing values in target: 0


## 6. Feature / Target Separation

Separate feature matrix $X$ and target vector $y$:
- Completely remove target column `Churn` from $X$.
- Exclude `customerID` from $X$: Customer ID is an arbitrary alphanumeric identifier without predictive business information; including it would introduce high-cardinality noise or leakage.

In [4]:
X, y = separate_features_target(df_clean, target_col="Churn", id_cols=["customerID"])

print(f"Feature matrix shape X: {X.shape}")
print(f"Target vector shape   y: {y.shape}")
print(f"Target column in X:      {'Churn' in X.columns}")
print(f"customerID in X:         {'customerID' in X.columns}")
print(f"Remaining features ({len(X.columns)}):\n", list(X.columns))

Feature matrix shape X: (7043, 19)
Target vector shape   y: (7043,)
Target column in X:      False
customerID in X:         False
Remaining features (19):
 ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


## 7. Identify Numerical and Categorical Features

We classify features programmatically into numerical and categorical lists using `identify_feature_types(X)`.

In [5]:
numerical_cols, categorical_cols = identify_feature_types(X)

print(f"Numerical features ({len(numerical_cols)}):")
for col in numerical_cols:
    print(f"  - {col} (dtype: {X[col].dtype}, unique: {X[col].nunique()})")

print(f"\nCategorical features ({len(categorical_cols)}):")
for col in categorical_cols:
    print(f"  - {col} (levels: {list(X[col].unique())})")

Numerical features (4):
  - SeniorCitizen (dtype: int64, unique: 2)
  - tenure (dtype: int64, unique: 73)
  - MonthlyCharges (dtype: float64, unique: 1585)
  - TotalCharges (dtype: float64, unique: 6531)

Categorical features (15):
  - gender (levels: ['Female', 'Male'])
  - Partner (levels: ['Yes', 'No'])
  - Dependents (levels: ['No', 'Yes'])
  - PhoneService (levels: ['No', 'Yes'])
  - MultipleLines (levels: ['No phone service', 'No', 'Yes'])
  - InternetService (levels: ['DSL', 'Fiber optic', 'No'])
  - OnlineSecurity (levels: ['No', 'Yes', 'No internet service'])
  - OnlineBackup (levels: ['Yes', 'No', 'No internet service'])
  - DeviceProtection (levels: ['No', 'Yes', 'No internet service'])
  - TechSupport (levels: ['No', 'Yes', 'No internet service'])
  - StreamingTV (levels: ['No', 'Yes', 'No internet service'])
  - StreamingMovies (levels: ['No', 'Yes', 'No internet service'])
  - Contract (levels: ['Month-to-month', 'One year', 'Two year'])
  - PaperlessBilling (levels: ['Ye

## 8. Train / Validation / Test Split

The dataset is partitioned into:
- **70% Training Set**: Used exclusively for parameter learning and fitting transformers.
- **15% Validation Set**: Used for model selection, threshold tuning, and hyperparameter optimization.
- **15% Test Set**: Held out untouched until final model evaluation.

We use a two-stage stratified split with `RANDOM_STATE = 42`:
1. Split $(X, y)$ into Train (70%) and Temp (30%), stratified by $y$.
2. Split $(X_{temp}, y_{temp})$ into Validation (50% of Temp = 15%) and Test (50% of Temp = 15%), stratified by $y_{temp}$.

In [6]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X,
    y,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    random_state=RANDOM_STATE,
)

n_total = len(df_clean)
print("Splitting complete.")
print(
    f"Train observations:      {len(X_train):,} ({len(X_train) / n_total * 100:.2f}%)"
)
print(f"Validation observations: {len(X_val):,} ({len(X_val) / n_total * 100:.2f}%)")
print(f"Test observations:       {len(X_test):,} ({len(X_test) / n_total * 100:.2f}%)")
print(f"Total observations:      {n_total:,} (100.00%)")

Splitting complete.
Train observations:      4,930 (70.00%)
Validation observations: 1,056 (14.99%)
Test observations:       1,057 (15.01%)
Total observations:      7,043 (100.00%)


## 9. Verify Split Distribution

We verify that the target churn distribution is preserved across all three splits.

In [7]:
split_summary = pd.DataFrame(
    [
        {
            "Partition": "Train",
            "Total Rows": len(y_train),
            "Percentage (%)": round(len(y_train) / n_total * 100, 2),
            "Churn = 0 (No)": f"{(y_train == 0).sum()} ({(y_train == 0).mean() * 100:.2f}%)",
            "Churn = 1 (Yes)": f"{(y_train == 1).sum()} ({(y_train == 1).mean() * 100:.2f}%)",
            "Churn Rate (%)": round(y_train.mean() * 100, 2),
        },
        {
            "Partition": "Validation",
            "Total Rows": len(y_val),
            "Percentage (%)": round(len(y_val) / n_total * 100, 2),
            "Churn = 0 (No)": f"{(y_val == 0).sum()} ({(y_val == 0).mean() * 100:.2f}%)",
            "Churn = 1 (Yes)": f"{(y_val == 1).sum()} ({(y_val == 1).mean() * 100:.2f}%)",
            "Churn Rate (%)": round(y_val.mean() * 100, 2),
        },
        {
            "Partition": "Test",
            "Total Rows": len(y_test),
            "Percentage (%)": round(len(y_test) / n_total * 100, 2),
            "Churn = 0 (No)": f"{(y_test == 0).sum()} ({(y_test == 0).mean() * 100:.2f}%)",
            "Churn = 1 (Yes)": f"{(y_test == 1).sum()} ({(y_test == 1).mean() * 100:.2f}%)",
            "Churn Rate (%)": round(y_test.mean() * 100, 2),
        },
        {
            "Partition": "Full Dataset",
            "Total Rows": len(y),
            "Percentage (%)": 100.00,
            "Churn = 0 (No)": f"{(y == 0).sum()} ({(y == 0).mean() * 100:.2f}%)",
            "Churn = 1 (Yes)": f"{(y == 1).sum()} ({(y == 1).mean() * 100:.2f}%)",
            "Churn Rate (%)": round(y.mean() * 100, 2),
        },
    ]
)

split_summary

Partition,Total Rows,Percentage (%),Churn = 0 (No),Churn = 1 (Yes),Churn Rate (%)
Train,4930,70.00,3622 (73.47%),1308 (26.53%),26.53
Validation,1056,14.99,776 (73.48%),280 (26.52%),26.52
Test,1057,15.01,776 (73.42%),281 (26.58%),26.58
Full Dataset,7043,100.00,5174 (73.46%),1869 (26.54%),26.54


## 10. Build Preprocessing Pipeline

We construct an unfitted `ColumnTransformer` with separate pipelines for numerical and categorical features:

```text
ColumnTransformer
│
├── Numerical Pipeline
│   ├── SimpleImputer(strategy='median')    → Outlier-resistant median imputation
│   └── StandardScaler()                    → Mean=0, Std=1 feature scaling
│
└── Categorical Pipeline
    ├── SimpleImputer(strategy='most_frequent') → Mode imputation
    └── OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        → Robust encoding that ignores unseen categories in validation/test/inference
        → Dense array output (lightweight memory footprint, directly model-compatible)
```

In [8]:
preprocessor = build_preprocessing_pipeline(
    numerical_cols=numerical_cols,
    categorical_cols=categorical_cols,
    verbose_feature_names=False,
)

print("Built ColumnTransformer successfully:")
print(preprocessor)

Built ColumnTransformer successfully:
ColumnTransformer(remainder='drop',
                  transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'Partner', 'Dependents',
                           

## 11. Fit Preprocessor on Training Data

> [!IMPORTANT]
> **DATA LEAKAGE PREVENTION RULE**: The preprocessing pipeline must **NEVER** see validation, test, or unsplit data during fitting. It learns medians, means, variances, and one-hot categories exclusively from $X_{train}$.

In [9]:
fit_preprocessor(preprocessor, X_train)

scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]
print("Fitting preprocessor exclusively on training data (rows: 4,930, columns: 19)...")
print("Preprocessor fitted successfully.\n")
print("Fitted StandardScaler means on training set:")
for col, mean_val in zip(numerical_cols, scaler.mean_):
    unit = " months" if col == "tenure" else ("$" if "Charges" in col else "")
    prefix = "$" if "Charges" in col else ""
    suffix = " months" if col == "tenure" else ""
    print(f"  - {col}: {prefix}{mean_val:.4f}{suffix}")

Fitting preprocessor exclusively on training data (rows: 4,930, columns: 19)...
Preprocessor fitted successfully.

Fitted StandardScaler means on training set:
  - SeniorCitizen:  0.1629
  - tenure:         32.4087 months
  - MonthlyCharges: $64.8837
  - TotalCharges:   $2288.7567


## 12. Transform Train / Validation / Test

Now we apply `transform()` across all partitions using the fitted preprocessor.

In [10]:
X_train_proc = transform_data(preprocessor, X_train, as_dataframe=True)
X_val_proc = transform_data(preprocessor, X_val, as_dataframe=True)
X_test_proc = transform_data(preprocessor, X_test, as_dataframe=True)

print("Transforming partitions...")
print(f"X_train_proc transformed: {X_train_proc.shape}")
print(f"X_val_proc transformed:   {X_val_proc.shape}")
print(f"X_test_proc transformed:  {X_test_proc.shape}")
print("All transformations completed successfully.")

Transforming partitions...
X_train_proc transformed: (4930, 45)
X_val_proc transformed:   (1056, 45)
X_test_proc transformed:  (1057, 45)
All transformations completed successfully.


## 13. Verify Transformed Data

We inspect transformed shapes, confirm zero missing values, and display a sample snippet of the transformed feature matrix.

In [11]:
print("Missing values check:")
print(f"  - Train nulls: {X_train_proc.isnull().sum().sum()}")
print(f"  - Val nulls:   {X_val_proc.isnull().sum().sum()}")
print(f"  - Test nulls:  {X_test_proc.isnull().sum().sum()}")

print("\nSample transformed features (5 observations, first 8 features):")
X_train_proc.iloc[:5, :8].round(4)

Missing values check:
  - Train nulls: 0
  - Val nulls:   0
  - Test nulls:  0

Sample transformed features (5 observations, first 8 features):


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes
185,-0.4411,-1.2801,-1.3323,-1.0028,1.0,0.0,0.0,1.0
2715,-0.4411,0.3501,-1.3190,-0.6121,0.0,1.0,1.0,0.0
3825,-0.4411,0.7984,-1.5132,-0.5735,1.0,0.0,0.0,1.0
1807,-0.4411,-1.2801,0.3806,-0.9760,1.0,0.0,1.0,0.0
132,-0.4411,1.4097,-0.4791,0.5408,0.0,1.0,1.0,0.0


## 14. Transformed Feature Names

We inspect all 45 feature names output by `preprocessor.get_feature_names_out()`.

In [12]:
feature_names = get_feature_names(preprocessor)
print(f"Total transformed features: {len(feature_names)}\n")
print("Numerical features (4):\n ", feature_names[:4])
print("\nOne-hot categorical features (41):\n ", feature_names[4:])

Total transformed features: 45

Numerical features (4):
  ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

One-hot categorical features (41):
  ['gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaperlessBilli

## 15. Save Preprocessing Artifact

We serialize the fitted `preprocessor` to `models/preprocessor.joblib` and save the processed train, validation, and test datasets to `data/processed/`.

In [13]:
# 1. Save fitted ColumnTransformer
saved_preprocessor_path = save_preprocessor(preprocessor, PREPROCESSOR_PATH)
print(f"Preprocessor artifact serialized to: {saved_preprocessor_path}")

# 2. Save processed datasets
saved_files = save_processed_datasets(
    X_train_proc,
    y_train,
    X_val_proc,
    y_val,
    X_test_proc,
    y_test,
    output_dir=PROCESSED_DATA_DIR,
)
print(
    f"Saved processed train partition:    data/processed/train.csv ({len(X_train_proc):,} rows, {X_train_proc.shape[1] + 1} cols)"
)
print(
    f"Saved processed validation partition: data/processed/val.csv ({len(X_val_proc):,} rows, {X_val_proc.shape[1] + 1} cols)"
)
print(
    f"Saved processed test partition:       data/processed/test.csv ({len(X_test_proc):,} rows, {X_test_proc.shape[1] + 1} cols)"
)
print("Saved dataset metadata:               data/processed/metadata.json")

# 3. Verify artifact deserialization
loaded_preprocessor = load_preprocessor(PREPROCESSOR_PATH)
reloaded_test_out = transform_data(loaded_preprocessor, X_test, as_dataframe=False)
original_test_out = transform_data(preprocessor, X_test, as_dataframe=False)
assert np.allclose(reloaded_test_out, original_test_out)
print("Reloaded preprocessor verified: identical test transformations.")

Preprocessor artifact serialized to: /home/lenovo/Documents/customer churn prediction/models/preprocessor.joblib
Saved processed train partition:    data/processed/train.csv (4,930 rows, 46 cols)
Saved processed validation partition: data/processed/val.csv (1,056 rows, 46 cols)
Saved processed test partition:       data/processed/test.csv (1,057 rows, 46 cols)
Saved dataset metadata:               data/processed/metadata.json
Reloaded preprocessor verified: identical test transformations.


## 16. Data Leakage Prevention

### What is Data Leakage?
Data leakage occurs when information from outside the training dataset (such as validation, test, or future operational data) is inadvertently incorporated into the feature engineering, scaling, or model fitting process. Leakage produces overly optimistic validation metrics that collapse when deployed to production.

### How This Pipeline Guarantees Zero Leakage:

1. **Partitioning Before Fitting**:
   The raw data is deterministically split into 70% Train, 15% Validation, and 15% Test *before* any statistical transformer is instantiated or fitted.

2. **Training-Only Imputation**:
   The numerical imputer (`SimpleImputer(strategy='median')`) calculates the median strictly from $X_{train}$. If $X_{val}$ contains missing values, it is imputed using the training median.

3. **Training-Only Scaling**:
   The `StandardScaler` calculates mean (mu) and standard deviation (sigma) strictly from X_train. X_val and X_test are transformed using z = (x - mu_train) / sigma_train.

4. **Training-Only Categorical Encoding**:
   `OneHotEncoder` discovers categories strictly from $X_{train}$. Unseen categories in validation, test, or production are safely ignored (`handle_unknown='ignore'`) rather than raising errors or mutating category mappings.

5. **Untouched Test Partition**:
   The test partition is transformed once and locked away. It will never be accessed during model selection, threshold tuning, or hyperparameter optimization in Step 5 and Step 6.

## 17. Preprocessing Summary & Next Steps

### Preprocessing Summary:
- **Raw records**: 7,043 observations, 21 columns (strictly immutable).
- **Deterministic cleaning**: Coerced 11 whitespace `TotalCharges` records to `0.0` for new accounts with `tenure == 0`.
- **Identifiers removed**: `customerID` dropped.
- **Target prepared**: `Churn` mapped to binary ${0, 1}$ ($y_{train}$ churn rate: 26.53%).
- **Partitions created**:
  - Train: 4,930 observations (70.00%)
  - Validation: 1,056 observations (14.99%)
  - Test: 1,057 observations (15.01%)
- **Transformed feature count**: 45 continuous features (4 scaled numerical + 41 one-hot indicators).
- **Saved artifacts**:
  - `models/preprocessor.joblib`: Reusable fitted `ColumnTransformer`.
  - `data/processed/train.csv`: Clean model-ready training set.
  - `data/processed/val.csv`: Model selection validation set.
  - `data/processed/test.csv`: Final evaluation test set.
  - `data/processed/metadata.json`: Complete partition metadata schema.

### Next Steps (Step 5):
- **Step 5 — Baseline Model: Logistic Regression**:
  - Train baseline Dummy Classifier and Logistic Regression model on `train.csv`.
  - Evaluate performance on `val.csv` using ROC-AUC, PR-AUC, and F1-score.
  - Keep `test.csv` strictly untouched until final model selection.